# Notebook 07 — Additional Data Verification and Multimodal Feature Inventory

## Objective
Verify newly downloaded PPMI imaging-derived tabular files and assess whether they can be used as additional baseline predictors for future multimodal machine learning.

## Scientific Background
The clinical-only model from Notebook 05–06 showed limited predictive performance. This notebook evaluates whether DaTSCAN-derived quantitative/visual data and MRI-derived tabular features are available for participants in the primary analytic cohort.

## Dataset Verification
This notebook checks:
- CSV files inside the imaging ZIP package.
- Presence of `PATNO` and `EVENT_ID`.
- Overlap with the Notebook 02 primary analytic cohort.
- Baseline availability.
- Candidate imaging predictors.
- Missingness before any modeling.

## Expected Output
A set of inventory and quality-control tables saved under:

`MyDrive/PPMI_PD_Progression/outputs/notebook_07_imaging_feature_inventory/`

No machine learning is performed in this notebook.

In [ ]:
# ============================================================
# 01. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# 02. Imports and project paths
# ============================================================

from pathlib import Path
import zipfile
import os
import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

RANDOM_STATE = 42
ID_COL = "PATNO"
EVENT_COL = "EVENT_ID"

PROJECT_DIR = Path("/content/drive/MyDrive/PPMI_PD_Progression")

RAW_DIRS = [
    PROJECT_DIR / "data" / "raw",
    PROJECT_DIR / "data" / "raw" / "additional",
    PROJECT_DIR / "data" / "raw" / "imaging",
]

EXTRACT_DIR = PROJECT_DIR / "data" / "extracted" / "imaging_tabular_selected"
NB02_DIR = PROJECT_DIR / "outputs" / "notebook_02_cohort_outcome"
NB03_DIR = PROJECT_DIR / "outputs" / "notebook_03_baseline_predictors"
OUT_DIR = PROJECT_DIR / "outputs" / "notebook_07_imaging_feature_inventory"

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("EXTRACT_DIR:", EXTRACT_DIR)
print("OUT_DIR:", OUT_DIR)
print("Notebook 02 directory exists:", NB02_DIR.exists())
print("Notebook 03 directory exists:", NB03_DIR.exists())

## 03. Required file placement

Upload the imaging ZIP file to one of these folders:

- `MyDrive/PPMI_PD_Progression/data/raw/`
- `MyDrive/PPMI_PD_Progression/data/raw/additional/`
- `MyDrive/PPMI_PD_Progression/data/raw/imaging/`

Expected ZIP name can be exactly or similar to:

`PPMI_Imaging_Tabular_Selected_30Jun2026.zip`

In [ ]:
# ============================================================
# 03. Locate imaging ZIP file
# ============================================================

zip_candidates = []
for raw_dir in RAW_DIRS:
    if raw_dir.exists():
        zip_candidates.extend(list(raw_dir.glob("*Imaging*Tabular*Selected*.zip")))
        zip_candidates.extend(list(raw_dir.glob("*imaging*tabular*selected*.zip")))
        zip_candidates.extend(list(raw_dir.glob("*DATSCAN*.zip")))
        zip_candidates.extend(list(raw_dir.glob("*DaTSCAN*.zip")))

zip_candidates = sorted(set(zip_candidates))

print("ZIP candidates found:")
for z in zip_candidates:
    print("-", z)

if len(zip_candidates) == 0:
    raise FileNotFoundError(
        "No imaging tabular ZIP file found. Upload the ZIP to one of the RAW_DIRS listed above."
    )

IMAGING_ZIP = zip_candidates[0]
print("\nUsing imaging ZIP:", IMAGING_ZIP)

In [ ]:
# ============================================================
# 04. Extract imaging ZIP
# ============================================================

with zipfile.ZipFile(IMAGING_ZIP, "r") as z:
    z.extractall(EXTRACT_DIR)

csv_files = sorted(EXTRACT_DIR.glob("*.csv"))

print(f"CSV files extracted/found: {len(csv_files)}")
for f in csv_files:
    print("-", f.name)

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV files found after extracting imaging ZIP.")

In [ ]:
# ============================================================
# 05. Load primary analytic cohort from Notebook 02
# ============================================================

cohort_candidates = sorted(NB02_DIR.glob("*primary*analytic*cohort*.csv")) + sorted(NB02_DIR.glob("*analytic*cohort*.csv"))

print("Analytic cohort candidates:")
for f in cohort_candidates:
    print("-", f.name)

if len(cohort_candidates) == 0:
    raise FileNotFoundError(
        "No analytic cohort CSV found in Notebook 02 output directory. "
        "Run Notebook 02 first or verify the path."
    )

# Prefer recommended-window cohort if present
preferred = [f for f in cohort_candidates if "recommended" in f.name.lower()]
COHORT_FILE = preferred[0] if preferred else cohort_candidates[0]

cohort = pd.read_csv(COHORT_FILE)
print("\nUsing cohort file:", COHORT_FILE.name)
print("Cohort shape:", cohort.shape)

if ID_COL not in cohort.columns:
    raise ValueError(f"{ID_COL} not found in analytic cohort.")

cohort_ids = set(cohort[ID_COL].dropna().astype(str))
print("Unique PATNO in primary analytic cohort:", len(cohort_ids))

cohort.head()

In [ ]:
# ============================================================
# 06. Read imaging CSVs and create file inventory
# ============================================================

def read_csv_safely(path):
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin1", low_memory=False)

datasets = {}
inventory_rows = []

for f in csv_files:
    df = read_csv_safely(f)
    datasets[f.stem] = df

    patno_present = ID_COL in df.columns
    event_present = EVENT_COL in df.columns

    unique_patno = df[ID_COL].nunique() if patno_present else np.nan
    unique_event = df[EVENT_COL].nunique() if event_present else np.nan

    inventory_rows.append({
        "dataset": f.stem,
        "file_name": f.name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "has_PATNO": patno_present,
        "has_EVENT_ID": event_present,
        "unique_PATNO": unique_patno,
        "unique_EVENT_ID": unique_event,
        "first_20_columns": "; ".join(list(df.columns[:20]))
    })

file_inventory = pd.DataFrame(inventory_rows).sort_values(["has_PATNO", "rows"], ascending=[False, False])
file_inventory.to_csv(OUT_DIR / "01_imaging_file_inventory.csv", index=False)
file_inventory

In [ ]:
# ============================================================
# 07. Overlap with primary analytic cohort
# ============================================================

overlap_rows = []

for name, df in datasets.items():
    if ID_COL not in df.columns:
        overlap_rows.append({
            "dataset": name,
            "status": "NO_PATNO",
            "rows": df.shape[0],
            "unique_PATNO": np.nan,
            "overlap_PATNO_with_primary_cohort": np.nan,
            "overlap_pct_of_primary_cohort": np.nan,
            "baseline_rows_BL": np.nan,
            "baseline_unique_PATNO_BL": np.nan,
            "baseline_overlap_PATNO_BL": np.nan,
            "baseline_overlap_pct_primary_cohort_BL": np.nan,
            "screening_rows_SC": np.nan,
            "screening_unique_PATNO_SC": np.nan,
            "screening_overlap_PATNO_SC": np.nan,
            "screening_overlap_pct_primary_cohort_SC": np.nan,
        })
        continue

    df_patno = df[ID_COL].dropna().astype(str)
    df_ids = set(df_patno)
    overlap_ids = df_ids.intersection(cohort_ids)

    if EVENT_COL in df.columns:
        event_str = df[EVENT_COL].astype(str)
        bl = df[event_str.eq("BL")]
        sc = df[event_str.eq("SC")]
        bl_ids = set(bl[ID_COL].dropna().astype(str))
        sc_ids = set(sc[ID_COL].dropna().astype(str))
    else:
        bl = pd.DataFrame()
        sc = pd.DataFrame()
        bl_ids = set()
        sc_ids = set()

    overlap_rows.append({
        "dataset": name,
        "status": "OK",
        "rows": df.shape[0],
        "unique_PATNO": len(df_ids),
        "overlap_PATNO_with_primary_cohort": len(overlap_ids),
        "overlap_pct_of_primary_cohort": 100 * len(overlap_ids) / len(cohort_ids),
        "baseline_rows_BL": bl.shape[0] if EVENT_COL in df.columns else np.nan,
        "baseline_unique_PATNO_BL": len(bl_ids) if EVENT_COL in df.columns else np.nan,
        "baseline_overlap_PATNO_BL": len(bl_ids.intersection(cohort_ids)) if EVENT_COL in df.columns else np.nan,
        "baseline_overlap_pct_primary_cohort_BL": 100 * len(bl_ids.intersection(cohort_ids)) / len(cohort_ids) if EVENT_COL in df.columns else np.nan,
        "screening_rows_SC": sc.shape[0] if EVENT_COL in df.columns else np.nan,
        "screening_unique_PATNO_SC": len(sc_ids) if EVENT_COL in df.columns else np.nan,
        "screening_overlap_PATNO_SC": len(sc_ids.intersection(cohort_ids)) if EVENT_COL in df.columns else np.nan,
        "screening_overlap_pct_primary_cohort_SC": 100 * len(sc_ids.intersection(cohort_ids)) / len(cohort_ids) if EVENT_COL in df.columns else np.nan,
    })

overlap_summary = pd.DataFrame(overlap_rows).sort_values(
    "overlap_pct_of_primary_cohort", ascending=False, na_position="last"
)
overlap_summary.to_csv(OUT_DIR / "02_imaging_overlap_with_primary_cohort.csv", index=False)
overlap_summary

In [ ]:
# ============================================================
# 08. Candidate imaging predictor detection
# ============================================================

EXCLUDE_PATTERNS = [
    r"^REC_ID$", r"^PATNO$", r"^EVENT_ID$", r"^SUB_EVENT_ID$", r"^PAG_NAME$",
    r"DATE", r"TIME", r"STAMP", r"UPDATE", r"PROTOCOL", r"IMAGEID",
    r"LIGAND", r"REASON", r"SPECIFY", r"PREVIOUSLY", r"ACQUIRED",
    r"ANALYZED", r"NOT_ANALYZED", r"ORIG_ENTRY", r"LAST_UPDATE",
    r"INFODT", r"MRIDATE", r"DATSCAN_DATE", r"DATSCAN_START_TIME",
]

def is_excluded_col(col):
    col_upper = str(col).upper()
    return any(re.search(p, col_upper) for p in EXCLUDE_PATTERNS)

candidate_rows = []

for name, df in datasets.items():
    for col in df.columns:
        if is_excluded_col(col):
            continue

        s = df[col]
        numeric_s = pd.to_numeric(s, errors="coerce")
        numeric_fraction = numeric_s.notna().mean()

        if numeric_fraction >= 0.50:
            inferred_type = "numeric"
            non_missing = int(numeric_s.notna().sum())
        else:
            inferred_type = "categorical_or_text"
            non_missing = int(s.notna().sum())

        if non_missing == 0:
            continue

        candidate_rows.append({
            "dataset": name,
            "variable": col,
            "inferred_type": inferred_type,
            "numeric_fraction": round(float(numeric_fraction), 4),
            "non_missing_total_rows": non_missing,
            "missing_pct_total_rows": round(100 * (1 - non_missing / len(df)), 2) if len(df) else np.nan,
            "unique_values": s.nunique(dropna=True),
            "recommended_for_baseline_inventory": inferred_type == "numeric"
        })

candidate_dictionary = pd.DataFrame(candidate_rows).sort_values(
    ["dataset", "recommended_for_baseline_inventory", "missing_pct_total_rows"],
    ascending=[True, False, True]
)

candidate_dictionary.to_csv(OUT_DIR / "03_imaging_candidate_predictor_dictionary.csv", index=False)
candidate_dictionary.head(50)

In [ ]:
# ============================================================
# 09. Build baseline imaging availability table for primary cohort
# ============================================================

def make_baseline_one_row_per_patno(df, dataset_name):
    df = df.copy()
    if ID_COL not in df.columns:
        return None

    # Use BL first. If no BL rows exist, use SC. If no EVENT_ID, use all rows.
    if EVENT_COL in df.columns:
        df[EVENT_COL] = df[EVENT_COL].astype(str)
        if (df[EVENT_COL] == "BL").any():
            base = df[df[EVENT_COL] == "BL"].copy()
            event_source = "BL"
        elif (df[EVENT_COL] == "SC").any():
            base = df[df[EVENT_COL] == "SC"].copy()
            event_source = "SC"
        else:
            return None
    else:
        base = df.copy()
        event_source = "NO_EVENT_ID"

    # Restrict to primary analytic cohort
    base[ID_COL] = base[ID_COL].astype(str)
    base = base[base[ID_COL].isin(cohort_ids)].copy()

    if base.empty:
        return None

    # Candidate numeric columns only for baseline feature matrix
    usable_cols = []
    for col in base.columns:
        if is_excluded_col(col):
            continue
        numeric_s = pd.to_numeric(base[col], errors="coerce")
        if numeric_s.notna().mean() >= 0.50:
            usable_cols.append(col)

    if len(usable_cols) == 0:
        return None

    tmp = base[[ID_COL] + usable_cols].copy()
    for col in usable_cols:
        tmp[col] = pd.to_numeric(tmp[col], errors="coerce")

    # Aggregate duplicates by participant using mean for numeric predictors
    agg = tmp.groupby(ID_COL, as_index=False)[usable_cols].mean()
    agg = agg.rename(columns={col: f"{dataset_name}__{col}" for col in usable_cols})
    agg["__event_source__"] = event_source
    return agg

baseline_feature_tables = []
baseline_avail_rows = []

for name, df in datasets.items():
    base = make_baseline_one_row_per_patno(df, name)
    if base is None:
        baseline_avail_rows.append({
            "dataset": name,
            "baseline_table_created": False,
            "event_source": None,
            "participants_with_any_baseline_feature": 0,
            "pct_primary_cohort_with_any_baseline_feature": 0,
            "candidate_numeric_features": 0,
            "features_missing_le_30pct": 0,
            "features_missing_le_50pct": 0,
        })
        continue

    event_source = base["__event_source__"].iloc[0]
    base = base.drop(columns=["__event_source__"])
    baseline_feature_tables.append(base)

    feature_cols = [c for c in base.columns if c != ID_COL]
    participants_any = base[feature_cols].notna().any(axis=1).sum()
    missing_pct = base[feature_cols].isna().mean().mul(100)

    baseline_avail_rows.append({
        "dataset": name,
        "baseline_table_created": True,
        "event_source": event_source,
        "participants_with_any_baseline_feature": int(participants_any),
        "pct_primary_cohort_with_any_baseline_feature": round(100 * participants_any / len(cohort_ids), 2),
        "candidate_numeric_features": len(feature_cols),
        "features_missing_le_30pct": int((missing_pct <= 30).sum()),
        "features_missing_le_50pct": int((missing_pct <= 50).sum()),
    })

baseline_availability = pd.DataFrame(baseline_avail_rows).sort_values(
    ["baseline_table_created", "pct_primary_cohort_with_any_baseline_feature"],
    ascending=[False, False]
)
baseline_availability.to_csv(OUT_DIR / "04_baseline_imaging_predictor_availability.csv", index=False)
baseline_availability

In [ ]:
# ============================================================
# 10. Merge candidate baseline imaging features with cohort IDs
# ============================================================

feature_matrix = cohort[[ID_COL]].copy()
feature_matrix[ID_COL] = feature_matrix[ID_COL].astype(str)

for base in baseline_feature_tables:
    base = base.copy()
    base[ID_COL] = base[ID_COL].astype(str)
    feature_matrix = feature_matrix.merge(base, on=ID_COL, how="left")

feature_cols = [c for c in feature_matrix.columns if c != ID_COL]
missing_summary = pd.DataFrame({
    "feature": feature_cols,
    "missing_n": [feature_matrix[c].isna().sum() for c in feature_cols],
    "missing_pct": [100 * feature_matrix[c].isna().mean() for c in feature_cols],
    "non_missing_n": [feature_matrix[c].notna().sum() for c in feature_cols],
}).sort_values("missing_pct")

feature_matrix.to_csv(OUT_DIR / "05_candidate_baseline_imaging_feature_matrix.csv", index=False)
missing_summary.to_csv(OUT_DIR / "06_candidate_baseline_imaging_feature_missingness.csv", index=False)

print("Candidate imaging feature matrix shape:", feature_matrix.shape)
print("Features with missingness <=30%:", int((missing_summary["missing_pct"] <= 30).sum()))
print("Features with missingness <=50%:", int((missing_summary["missing_pct"] <= 50).sum()))
missing_summary.head(30)

In [ ]:
# ============================================================
# 11. Recommended imaging feature sets
# ============================================================

recommended_rows = []

# Dataset-level recommendations
for _, row in baseline_availability.iterrows():
    if not row["baseline_table_created"]:
        decision = "Not recommended for baseline model"
        reason = "No usable baseline numeric table created for primary cohort."
    elif row["pct_primary_cohort_with_any_baseline_feature"] >= 70 and row["features_missing_le_30pct"] >= 1:
        decision = "Strong candidate for primary multimodal model"
        reason = "Good participant coverage and at least one feature with <=30% missingness."
    elif row["pct_primary_cohort_with_any_baseline_feature"] >= 40 and row["features_missing_le_50pct"] >= 1:
        decision = "Candidate for sensitivity/multimodal exploratory model"
        reason = "Moderate coverage or higher missingness; may reduce sample size."
    else:
        decision = "Not recommended for primary model"
        reason = "Low overlap with primary analytic cohort or excessive missingness."

    recommended_rows.append({
        "dataset": row["dataset"],
        "event_source": row["event_source"],
        "participant_coverage_pct": row["pct_primary_cohort_with_any_baseline_feature"],
        "candidate_numeric_features": row["candidate_numeric_features"],
        "features_missing_le_30pct": row["features_missing_le_30pct"],
        "features_missing_le_50pct": row["features_missing_le_50pct"],
        "decision": decision,
        "reason": reason,
    })

recommended_feature_sets = pd.DataFrame(recommended_rows).sort_values(
    ["decision", "participant_coverage_pct"], ascending=[True, False]
)
recommended_feature_sets.to_csv(OUT_DIR / "07_recommended_imaging_feature_sets.csv", index=False)
recommended_feature_sets

In [ ]:
# ============================================================
# 12. Quality Control Checklist
# ============================================================

qc_items = []

def add_qc(item, status, details):
    qc_items.append({"qc_item": item, "status": status, "details": details})

add_qc(
    "Imaging ZIP found and extracted",
    "PASS" if len(csv_files) > 0 else "FAIL",
    f"{len(csv_files)} CSV files found."
)

add_qc(
    "Primary analytic cohort loaded",
    "PASS" if len(cohort_ids) > 0 else "FAIL",
    f"{len(cohort_ids)} primary analytic cohort participants."
)

add_qc(
    "At least one imaging table contains PATNO",
    "PASS" if any(v[ID_COL].nunique() for v in datasets.values() if ID_COL in v.columns) else "FAIL",
    f"{sum(ID_COL in df.columns for df in datasets.values())} datasets contain PATNO."
)

add_qc(
    "At least one imaging table contains EVENT_ID",
    "PASS" if sum(EVENT_COL in df.columns for df in datasets.values()) >= 1 else "FAIL",
    f"{sum(EVENT_COL in df.columns for df in datasets.values())} datasets contain EVENT_ID."
)

add_qc(
    "Candidate imaging baseline feature matrix created",
    "PASS" if len(feature_cols) > 0 else "FAIL",
    f"{len(feature_cols)} candidate imaging features."
)

add_qc(
    "Primary model not run in this notebook",
    "PASS",
    "This notebook performs inventory and availability checks only."
)

qc = pd.DataFrame(qc_items)
qc.to_csv(OUT_DIR / "08_quality_control_checklist.csv", index=False)
qc

In [ ]:
# ============================================================
# 13. Summary report
# ============================================================

report_lines = []
report_lines.append("Notebook 07 — Additional Data Verification and Multimodal Feature Inventory")
report_lines.append("=" * 78)
report_lines.append(f"Imaging ZIP used: {IMAGING_ZIP}")
report_lines.append(f"CSV files found: {len(csv_files)}")
report_lines.append(f"Primary analytic cohort participants: {len(cohort_ids)}")
report_lines.append("")
report_lines.append("File inventory:")
for _, r in file_inventory.iterrows():
    report_lines.append(
        f"- {r['dataset']}: rows={r['rows']}, cols={r['columns']}, "
        f"PATNO={r['has_PATNO']}, EVENT_ID={r['has_EVENT_ID']}"
    )
report_lines.append("")
report_lines.append("Baseline imaging availability by dataset:")
for _, r in baseline_availability.iterrows():
    report_lines.append(
        f"- {r['dataset']}: created={r['baseline_table_created']}, "
        f"event_source={r['event_source']}, "
        f"coverage={r['pct_primary_cohort_with_any_baseline_feature']}%, "
        f"features<=30% missing={r['features_missing_le_30pct']}, "
        f"features<=50% missing={r['features_missing_le_50pct']}"
    )
report_lines.append("")
report_lines.append(f"Candidate imaging feature matrix shape: {feature_matrix.shape}")
report_lines.append(f"Features with <=30% missingness: {int((missing_summary['missing_pct'] <= 30).sum())}")
report_lines.append(f"Features with <=50% missingness: {int((missing_summary['missing_pct'] <= 50).sum())}")
report_lines.append("")
report_lines.append("Interpretation:")
report_lines.append(
    "This notebook verifies whether newly downloaded imaging-derived tabular data can be used "
    "as additional baseline predictors. No machine learning was performed."
)

report = "\n".join(report_lines)
(OUT_DIR / "09_notebook_07_summary_report.txt").write_text(report)

print(report)

## Quality Control Checklist

Before moving to Notebook 08, confirm:

- Imaging tabular ZIP was extracted successfully.
- `PATNO` is present in the main imaging tables.
- Baseline or screening imaging rows overlap with the Notebook 02 analytic cohort.
- Candidate imaging predictors have acceptable missingness.
- No machine learning was performed in this notebook.
- Final decision on whether imaging features are suitable is based on participant coverage and missingness.